In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import pickle
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/telco-customer-churn-data/synthetic_customer_churn_100k.csv


In [2]:
data = pd.read_csv("/kaggle/input/telco-customer-churn-data/synthetic_customer_churn_100k.csv")

In [3]:
data.head()

,CustomerID,Age,Gender,Tenure,MonthlyCharges,Contract,PaymentMethod,TotalCharges,Churn
0,1,56,Female,68,147.58,Two year,Bank transfer,10052.03,No
1,2,69,Male,32,22.54,Month-to-month,Mailed check,686.78,No
2,3,46,Female,10,52.47,One year,Electronic check,537.88,No
3,4,32,Male,22,109.67,Month-to-month,Mailed check,2390.04,Yes
4,5,60,Female,54,130.98,Month-to-month,Credit card,7081.28,No


In [4]:
data = data.drop(columns=["CustomerID"])

In [5]:
from sklearn.preprocessing import LabelEncoder

le_gender = LabelEncoder()
le_churn = LabelEncoder()

data['Gender'] = le_gender.fit_transform(data['Gender'])
data['Churn'] = le_churn.fit_transform(data['Churn'])

data

,Age,Gender,Tenure,MonthlyCharges,Contract,PaymentMethod,TotalCharges,Churn
0,56,0,68,147.58,Two year,Bank transfer,10052.03,0
1,69,1,32,22.54,Month-to-month,Mailed check,686.78,0
2,46,0,10,52.47,One year,Electronic check,537.88,0
3,32,1,22,109.67,Month-to-month,Mailed check,2390.04,1
4,60,0,54,130.98,Month-to-month,Credit card,7081.28,0
...,...,...,...,...,...,...,...,...
99995,31,1,49,26.07,Month-to-month,Electronic check,1220.50,0
99996,64,0,44,123.22,Month-to-month,Mailed check,5384.38,0
99997,48,2,32,75.37,Month-to-month,Credit card,2372.33,1
99998,42,0,60,114.00,Month-to-month,Mailed check,6826.55,0


In [6]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded = ohe.fit_transform(data[['Contract','PaymentMethod']])


In [7]:
encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(['Contract','PaymentMethod']),
    index=data.index
)


In [8]:
data = pd.concat([data, encoded_df], axis=1)

In [9]:
data.drop(['Contract','PaymentMethod'], axis=1, inplace=True)

In [10]:
print(data.head())

   Age  Gender  Tenure  MonthlyCharges  TotalCharges  Churn  \
0   56       0      68          147.58      10052.03      0   
1   69       1      32           22.54        686.78      0   
2   46       0      10           52.47        537.88      0   
3   32       1      22          109.67       2390.04      1   
4   60       0      54          130.98       7081.28      0   

   Contract_Month-to-month  Contract_One year  Contract_Two year  \
0                      0.0                0.0                1.0   
1                      1.0                0.0                0.0   
2                      0.0                1.0                0.0   
3                      1.0                0.0                0.0   
4                      1.0                0.0                0.0   

   PaymentMethod_Bank transfer  PaymentMethod_Credit card  \
0                          1.0                        0.0   
1                          0.0                        0.0   
2                          0.

In [11]:
## Save the encoders and sscaler
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(le_gender,file)

with open('label_encoder_churn.pkl','wb') as file:
    pickle.dump(le_churn,file)

with open('onehot_encoder.pkl','wb') as file:
    pickle.dump(ohe,file)

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = data.drop(['Churn'],axis=1)
y = data['Churn']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [13]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [14]:
X_train

array([[-1.43176326,  0.77816513, -0.94232825, ..., -0.50148394,
         1.36565796, -0.58035251],
       [-1.26672572,  0.77816513, -1.56742879, ..., -0.50148394,
         1.36565796, -0.58035251],
       [ 0.76873731,  2.53039466,  1.31765064, ..., -0.50148394,
         1.36565796, -0.58035251],
       ...,
       [-0.60657555,  0.77816513, -0.26914305, ..., -0.50148394,
         1.36565796, -0.58035251],
       [-1.48677577, -0.97406439,  0.64446544, ..., -0.50148394,
         1.36565796, -0.58035251],
       [-0.60657555,  0.77816513, -1.08658222, ..., -0.50148394,
        -0.73224777, -0.58035251]])

In [15]:
X_test

array([[-0.77161309, -0.97406439, -0.17297373, ...,  1.99408181,
        -0.73224777, -0.58035251],
       [ 1.53891251,  0.77816513, -0.36531236, ...,  1.99408181,
        -0.73224777, -0.58035251],
       [-0.99166315,  0.77816513,  1.02914269, ..., -0.50148394,
        -0.73224777,  1.72309068],
       ...,
       [-0.88163812, -0.97406439, -1.37509016, ..., -0.50148394,
        -0.73224777, -0.58035251],
       [-0.22148795,  0.77816513, -0.41339702, ..., -0.50148394,
        -0.73224777, -0.58035251],
       [ 0.27362468,  0.77816513, -1.37509016, ..., -0.50148394,
         1.36565796, -0.58035251]])

In [16]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime


## Build Our ANN Model
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), ## HL1 Connected wwith input layer
    Dense(32,activation='relu'), ## HL2
    Dense(1,activation='sigmoid')  ## output layer
]

)

2026-02-13 16:25:30.551403: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770999930.739090      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770999930.793688      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770999931.265310      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770999931.265349      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770999931.265352      55 computation_placer.cc:177] computation placer alr

In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

<LossFunctionWrapper(<function binary_crossentropy at 0x7d86dc87dd00>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>

In [19]:
## compile the model
model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])

In [20]:
## Set up the Tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [21]:
## Set up Early Stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [22]:
### Train the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100


I0000 00:00:1770999975.406866     126 service.cc:152] XLA service 0x7d860c004730 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770999975.406902     126 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1770999975.406905     126 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1770999975.693059     126 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1770999976.444845     126 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2500/2500 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.7313 - loss: 0.5038 - val_accuracy: 0.7459 - val_loss: 0.4789
Epoch 2/100
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7483 - loss: 0.4740 - val_accuracy: 0.7493 - val_loss: 0.4741
Epoch 3/100
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7509 - loss: 0.4721 - val_accuracy: 0.7519 - val_loss: 0.4729
Epoch 4/100
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.7505 - loss: 0.4693 - val_accuracy: 0.7533 - val_loss: 0.4693
Epoch 5/100
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.7533 - loss: 0.4667 - val_accuracy: 0.7509 - val_loss: 0.4698
Epoch 6/100
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.7512 - loss: 0.4712 - val_accuracy: 0.7489 - val_loss: 0.4721
Epoch 7/100
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7547 - loss: 0.4684 - val_accuracy: 0.7502 - val_loss: 0.4697
Epoch 8/100
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7495 - loss: 0.4717 - val

In [23]:
model.save('model.h5')

In [ ]:
## PREDICTING THE MODEL OUTPUT

In [44]:
input_data = pd.DataFrame({
    'Age':[25],
    'Gender':['Male'],
    'Tenure':[2],
    'MonthlyCharges':[120],
    'Contract':['Month-to-month'],
    'PaymentMethod':['Electronic check'],
    'TotalCharges':[240]
})


In [45]:
input_data['Gender'] = le_gender.transform(input_data['Gender'])

In [46]:
encoded = ohe.transform(input_data[['Contract','PaymentMethod']])

encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(['Contract','PaymentMethod'])
)


In [47]:
input_data = input_data.drop(['Contract','PaymentMethod'], axis=1)


In [48]:
input_data = pd.concat([input_data.reset_index(drop=True), encoded_df], axis=1)


In [49]:
scaled_input = scaler.transform(input_data)


In [50]:
prediction = model.predict(scaled_input)

print("Raw prediction:", prediction)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Raw prediction: [[0.99988526]]


In [51]:
pred_binary = (prediction > 0.5).astype(int)
print("Binary prediction:", pred_binary)


Binary prediction: [[1]]


In [52]:
churn_result = le_churn.inverse_transform(pred_binary.flatten())
print("Final Prediction:", churn_result[0])


Final Prediction: Yes
